# ML Zoomcamp 2026 – Homework 1: Car Fuel Efficiency

**Autor:** [Angel Uriel Monterrosas Gonzalez]

**Fecha:** [Lunes 28 de septiembre 2026]

**Módulo:** 01 – Introduction to Machine Learning

**Actividad:** Homework 01

**Dataset:** [car_fuel_efficiency.csv](https://raw.githubusercontent.com/alexeygrigorev/datasets/master/car_fuel_efficiency.csv)

Este notebook responde a las preguntas del homework oficial disponible en:
https://courses.datatalks.club/ml-zoomcamp-2026/homework/hw01#%%

## Objetivos de aprendizaje

Al finalizar este notebook serás capaz de:

- Configurar un entorno reproducible con versiones de librerías y semillas.
- Cargar y describir un dataset tabular con Pandas.
- Realizar un Análisis Exploratorio de Datos (EDA) reproducible.
- Identificar y tratar valores faltantes de forma justificada.
- Aplicar operaciones matriciales básicas con NumPy.
- Responder preguntas de negocio a partir de los datos.
- Documentar hallazgos y limitaciones de forma clara.

## 3. Setup

Importamos las librerías necesarias, fijamos semillas para reproducibilidad
y definimos la ruta del dataset.

In [50]:
# 3.1 Imports
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# 3.2 Versiones
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

# 3.3 Semilla global
SEED = 42
np.random.seed(SEED)

# 3.4 Ruta del dataset (URL directa)
DATASET_URL = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/refs/heads/main/cohorts/2026/data/car_fuel_efficiency_2026.csv"

Pandas version: 3.0.6
NumPy version: 2.5.3


## 4. Carga y descripción de datos

Cargamos el CSV directamente desde la URL y exploramos su estructura.

In [51]:
# 4.1 Carga
df = pd.read_csv(DATASET_URL)

# 4.2 Dimensiones
print("Shape:", df.shape)

# 4.3 Primeras filas
df.head()


Shape: (10000, 11)


,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [53]:
# 4.4 Información general
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   model_year           10000 non-null  int64  
 1   origin               10000 non-null  str    
 2   fuel_type            10000 non-null  str    
 3   drivetrain           10000 non-null  str    
 4   num_doors            10000 non-null  int64  
 5   engine_displacement  10000 non-null  int64  
 6   num_cylinders        10000 non-null  int64  
 7   horsepower           9123 non-null   float64
 8   vehicle_weight       10000 non-null  int64  
 9   acceleration         9736 non-null   float64
 10  fuel_efficiency_mpg  10000 non-null  float64
dtypes: float64(3), int64(5), str(3)
memory usage: 859.5 KB


In [54]:
# 4.5 Estadísticas descriptivas
df.describe(include="all")

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
count,10000.000000,10000,10000,10000,10000.00000,10000.000000,10000.000000,9123.000000,10000.000000,9736.000000,10000.000000
unique,NaN,3,3,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,USA,Gasoline,Front-wheel drive,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,4202,6913,5519,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,1999.549500,NaN,NaN,NaN,3.50400,2268.807000,6.035200,254.446016,4286.754000,18.888044,29.997700
std,14.207925,NaN,NaN,NaN,0.95209,183.447355,0.542578,22.844613,266.212491,1.279543,2.930245
min,1975.000000,NaN,NaN,NaN,2.00000,1590.000000,4.000000,171.000000,3320.000000,14.000000,19.800000
25%,1987.000000,NaN,NaN,NaN,3.00000,2150.000000,6.000000,239.000000,4110.000000,18.000000,28.000000
50%,2000.000000,NaN,NaN,NaN,4.00000,2270.000000,6.000000,254.000000,4280.000000,18.900000,30.000000
75%,2012.000000,NaN,NaN,NaN,4.00000,2390.000000,6.000000,270.000000,4470.000000,19.700000,31.900000


## 5. EDA reproducible

Analizamos la distribución de variables clave y detectamos valores faltantes.

In [56]:
# 5.1 Tipos de combustible
print("Fuel types:", df["fuel_type"].nunique())
print(df["fuel_type"].value_counts())

Fuel types: 3
fuel_type
Gasoline    6913
Diesel      2053
Hybrid      1034
Name: count, dtype: int64


In [57]:
# 5.2 Valores faltantes por columna
missing = df.isna().sum()
missing_cols = missing[missing > 0]
print("Columnas con valores faltantes:", len(missing_cols))
print(missing_cols)

Columnas con valores faltantes: 2
horsepower      877
acceleration    264
dtype: int64


In [58]:
# 5.3 Distribución de origen
print(df["origin"].value_counts())

origin
USA       4202
Asia      2994
Europe    2804
Name: count, dtype: int64


In [59]:
# 5.4 Eficiencia de combustible por origen
df.groupby("origin")["fuel_efficiency_mpg"].describe()

,count,mean,std,min,25%,50%,75%,max
origin,,,,,,,,
Asia,2994.0,30.741015,2.835474,21.3,28.700,30.7,32.5,41.2
Europe,2804.0,30.552675,2.790667,20.7,28.675,30.5,32.3,39.4
USA,4202.0,29.097739,2.846583,19.8,27.100,29.0,31.0,38.2


## 6. Feature engineering

Para este homework introductorio no se requiere crear nuevas variables,
pero documentamos posibles transformaciones futuras (ej. codificación de
variables categóricas, escalado).

In [ ]:
# 6.1 Ejemplo: codificación one-hot (opcional, no necesaria para las preguntas)
# df_encoded = pd.get_dummies(df, columns=["origin", "fuel_type"], drop_first=True)
# df_encoded.head()

## 7. Baseline

Establecemos un baseline trivial: predecir la media de `fuel_efficiency_mpg`.
Esto nos da una referencia mínima de rendimiento.

In [61]:
# 7.1 Baseline: media
baseline_pred = df["fuel_efficiency_mpg"].mean()
baseline_rmse = np.sqrt(mean_squared_error(df["fuel_efficiency_mpg"], [baseline_pred]*len(df)))
print(f"Baseline (media) RMSE: {baseline_rmse:.2f}")

Baseline (media) RMSE: 2.93


## 8. Modelado e iteraciones

Entrenamos una regresión lineal simple usando `horsepower` para predecir
`fuel_efficiency_mpg`. Antes, tratamos los valores faltantes de `horsepower`
con la moda (valor más frecuente), tal como pide el homework.

In [62]:
# 8.1 Mediana inicial de horsepower
median_before = df["horsepower"].median()
print(f"Mediana antes de imputar: {median_before}")

# 8.2 Moda de horsepower
mode_hp = df["horsepower"].mode()[0]
print(f"Moda de horsepower: {mode_hp}")

# 8.3 Imputación con la moda
df["horsepower"] = df["horsepower"].fillna(mode_hp)

# 8.4 Mediana después de imputar
median_after = df["horsepower"].median()
print(f"Mediana después de imputar: {median_after}")

# 8.5 ¿Cambió?
if median_after > median_before:
    print("La mediana aumentó.")
elif median_after < median_before:
    print("La mediana disminuyó.")
else:
    print("La mediana no cambió.")

Mediana antes de imputar: 254.0
Moda de horsepower: 252.0
Mediana después de imputar: 252.0
La mediana disminuyó.


In [63]:
# 8.6 Regresión lineal simple
X = df[["horsepower"]].values
y = df["fuel_efficiency_mpg"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.2f}")
print(f"R²: {r2_score(y_test, y_pred):.2f}")

RMSE: 2.86
R²: 0.08


## 9. Evaluación

Calculamos métricas sobre el conjunto de prueba y comparamos con el baseline.

In [64]:
# 9.1 Comparación con baseline
print(f"Baseline RMSE: {baseline_rmse:.2f}")
print(f"Modelo RMSE:   {np.sqrt(mean_squared_error(y_test, y_pred)):.2f}")

Baseline RMSE: 2.93
Modelo RMSE:   2.86


## 10. Interpretación de resultados

Respondemos a las preguntas del homework 2026 (dataset `car_fuel_efficiency_2026.csv`):

- **P1 (Pandas version):** `3.0.6`
- **P2 (Records count):** `10000` registros → opción **10000**
- **P3 (Fuel types):** `3` tipos (Gasoline, Diesel, Hybrid) → opción **3**
- **P4 (Missing values):** `2` columnas con valores faltantes (`horsepower`, `acceleration`) → opción **2**
- **P5 (Max fuel efficiency – Asia):** `41.2` mpg → opción **41.2**
- **P6 (Median horsepower):** pasó de `254.0` a `252.0` tras imputar con la moda (`252.0`) → **Yes, it decreased**
- **P7 (Sum of weights):** `w.sum() = 0.3692` → opción **0.369**


In [65]:
# 10.1 Cálculo matricial para Pregunta 7
# Seleccionar autos de Asia
df_asia = df[df["origin"] == "Asia"].copy()

# Seleccionar columnas y primeras 7 filas
X_asia = df_asia[["vehicle_weight", "model_year"]].iloc[:7].to_numpy()

# Matriz XTX
XTX = X_asia.T.dot(X_asia)
XTX_inv = np.linalg.inv(XTX)

# Vector y
y_vec = np.array([1100, 1300, 800, 900, 1000, 1100, 1200])

# Pesos
w = XTX_inv.dot(X_asia.T).dot(y_vec)
print(f"Suma de los elementos de w: {w.sum():.4f}")

Suma de los elementos de w: 0.3692


## 11. Conclusiones

- Se configuró un entorno reproducible con versiones fijas (Pandas 3.0.6, NumPy 2.5.3) y semilla global `SEED = 42`.
- El dataset `car_fuel_efficiency_2026.csv` contiene **10 000 registros** y **11 columnas** (8 numéricas + 3 categóricas).
- Se identificaron **3 tipos de combustible**: Gasoline (6913), Diesel (2053) y Hybrid (1034).
- Solo **2 columnas** presentan valores faltantes: `horsepower` (877 nulos) y `acceleration` (264 nulos).
- La máxima eficiencia de combustible para autos de origen **Asia** es **41.2 mpg**, la más alta entre las tres regiones.
- La imputación de `horsepower` con su moda (`252.0`) **disminuyó** la mediana de `254.0` a `252.0`.
- El baseline (predecir la media) obtiene un RMSE de **2.93**; la regresión lineal simple con `horsepower` lo mejora ligeramente hasta **2.86** (R² = 0.08). La mejora es marginal, lo que sugiere que `horsepower` por sí sola explica poco de la varianza en eficiencia.
- El cálculo matricial para la Pregunta 7 dio como resultado `w.sum() = 0.3692` (opción **0.369**).



## 12. Próximos pasos

- Probar modelos más complejos (Random Forest, Gradient Boosting).
- Ingeniería de características: interacciones, polinomios.
- Validación cruzada para una estimación más robusta.
- Análisis de residuos para diagnosticar el modelo.
- Documentar el pipeline en un script reproducible.

## 13. Referencias

- Dataset: [car_fuel_efficiency_2026.csv](https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/refs/heads/main/cohorts/2026/data/car_fuel_efficiency_2026.csv)
- Homework oficial: [ML Zoomcamp 2026 – HW01](https://courses.datatalks.club/ml-zoomcamp-2026/homework/hw01)
- Documentación de Pandas: https://pandas.pydata.org/docs/
- Documentación de NumPy: https://numpy.org/doc/
- Documentación de Scikit-learn: https://scikit-learn.org/stable/